# wcd_pipeline — Full Pipeline (Discovery → Validation → Catalog)
Part of the Public Webcam Discovery System.

## Pipeline execution model

The pipeline uses a **streaming parallel producer-consumer architecture** rather than running agents in sequence:

```
DirectoryAgent.stream()  ─┐
                          ├─► asyncio.Queue ──► ValidationAgent.run_from_queue()
SearchAgent.stream()     ─┘
```

1. **DirectoryAgent** and **SearchAgent** run **concurrently** — both start immediately and stream `CameraCandidate` objects into a shared bounded queue as they are found (batch-by-batch for directory crawling, city-by-city for search)
2. **ValidationAgent** starts processing **immediately** — it drains the queue in batches of 100 so HTTP probing and geocoding begin as soon as the first batch of candidates arrives, without waiting for all discovery to finish
3. **CatalogAgent** runs after all records are collected — deduplication requires the complete record set

### Why this matters
With a 25 s read timeout and up to 50 concurrent probes, waiting for full discovery before validation adds unnecessary serial latency.  The streaming model cuts total pipeline time roughly in half for large runs (e.g. Tier 1: ~40 min → ~20 min).

## Geocoding
Camera location strings are passed to an **Ollama LLM** (`USE_LLM_GEODECODE = True` by default).
The model returns structured `location`, `latitude`, `longitude`, `country`, `region`, and `continent`.
Results are cached process-wide so each unique location is resolved only once per pipeline run.
Set `WCD_USE_LLM_GEODECODE=false` (env var) to revert to the Nominatim fallback chain.

## Notebook vs full pipeline
This notebook runs agents **step-by-step** so you can inspect intermediate outputs at each stage.
To run the complete streaming pipeline with a single command, see **Step 2b** below.

In [1]:
# Environment setup — run this cell first every session
import subprocess, sys, os
from pathlib import Path

IN_COLAB     = "google.colab" in sys.modules
IN_SAGEMAKER = os.environ.get("SM_TRAINING_ENV") is not None

# ── 1. Locate / clone the project ────────────────────────────────────────────
if IN_COLAB:
    repo_dir = Path("/content/webcam-discovery")
    if not repo_dir.exists():
        subprocess.run(
            ["git", "clone", "https://github.com/dshipley71/webcam-discovery", str(repo_dir)],
            check=True,
        )
    os.chdir(repo_dir)
elif IN_SAGEMAKER:
    repo_dir = Path(os.environ.get("SM_MODEL_DIR", "/opt/ml/model")) / "webcam-discovery"
    if not repo_dir.exists():
        subprocess.run(
            ["git", "clone", "https://github.com/dshipley71/webcam-discovery", str(repo_dir)],
            check=True,
        )
    os.chdir(repo_dir)
else:
    # Local: assume CWD is already the project root (where pyproject.toml lives)
    repo_dir = Path.cwd()

print(f"Project root: {repo_dir}")

# ── 2. Install the package using the absolute path ───────────────────────────
# Using repo_dir directly avoids any CWD ambiguity (relative "." can resolve to
# the wrong directory if the kernel was restarted or chdir ran more than once).
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", f"{repo_dir}[notebooks]", "-q"],
    check=True,
)

# ── 3. Verify importability — fallback: insert src/ into sys.path ─────────────
# pip install -e works in a fresh kernel but can be lost after a kernel restart
# if the site-packages link is stale.  Inserting src/ ensures the package is
# always resolvable without a second install.
src_path = str(repo_dir / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import webcam_discovery  # noqa: F401  — raises ImportError if src layout is broken
from webcam_discovery.config import settings
settings.ensure_dirs()

print(f"webcam_discovery loaded from: {Path(webcam_discovery.__file__).parent}")
print("Ready ✓")

Project root: /content/webcam-discovery
webcam_discovery loaded from: /content/webcam-discovery/src/webcam_discovery
Ready ✓


## Step 1 — Ollama API key configuration

`GeoEnrichmentSkill` uses **Ollama cloud** for geocoding (`USE_LLM_GEODECODE = True` by default).
Supply your API key via **one** of these methods — highest priority wins:

| Method | How |
|--------|-----|
| **Colab secret** | Add a secret named `OLLAMA_API_KEY` in the Colab Secrets panel (🔑) |
| **Manual** | Set `OLLAMA_API_KEY = "sk-..."` in the cell below |
| **Env var** | Set `WCD_OLLAMA_API_KEY=<key>` in your shell or a `.env` file |

Leave `OLLAMA_API_KEY = ""` to connect without authentication (local Ollama or a public endpoint).

In [2]:
import os

# ── Ollama API key resolution ─────────────────────────────────────────────────
# Priority: Colab userdata secret → manual value below → already-set env var

_ollama_key = ""

# 1. Google Colab userdata secret (recommended for Colab sessions)
try:
    from google.colab import userdata  # type: ignore
    _ollama_key = userdata.get("OLLAMA_API_KEY") or ""
    if _ollama_key:
        print(f"Ollama API key loaded from Colab secrets ✓  (length={len(_ollama_key)})")
except Exception:
    pass

# 2. Manual fallback — paste your key here if not using Colab secrets
if not _ollama_key:
    OLLAMA_API_KEY = ""   # ← paste key here, e.g. "sk-ollama-..."
    _ollama_key = OLLAMA_API_KEY

# 3. Propagate to environment so settings (pydantic-settings) picks it up
if _ollama_key:
    os.environ["WCD_OLLAMA_API_KEY"] = _ollama_key
    if "Colab" not in (globals().get("_source", "") or ""):
        print(f"Ollama API key set from manual value ✓  (length={len(_ollama_key)})")
else:
    print("No Ollama API key set — connecting without authentication")
    print("  (fine for a local Ollama instance; cloud endpoints require a key)")

# ── Optional: override endpoint / model ──────────────────────────────────────
# Uncomment and edit to point at a local Ollama instance or a different model.
# os.environ["WCD_OLLAMA_BASE_URL"] = "http://localhost:11434"  # local Ollama
os.environ["WCD_OLLAMA_BASE_URL"] = "https://ollama.com"  # Ollama cloud (default)
os.environ["WCD_OLLAMA_MODEL"]    = "gemma3:27b"               # model (default)

# ── Confirm active settings ───────────────────────────────────────────────────
# Re-import settings so any env-var overrides above take effect.
from importlib import reload
import webcam_discovery.config as _cfg_mod
reload(_cfg_mod)
from webcam_discovery.config import settings  # noqa: F811 — intentional reload

print()
print(f"Geocoding mode  : {'LLM via Ollama' if settings.use_llm_geodecode else 'Nominatim (legacy)'}")
print(f"Ollama endpoint : {settings.ollama_base_url}")
print(f"Ollama model    : {settings.ollama_model}")
print(f"API key present : {'yes' if settings.ollama_api_key else 'no'}")

Ollama API key loaded from Colab secrets ✓  (length=57)
Ollama API key set from manual value ✓  (length=57)

Geocoding mode  : LLM via Ollama
Ollama endpoint : https://ollama.com
Ollama model    : gemma3:27b
API key present : yes


## Step 2a - Build environment

Install the package with dev dependencies so the pipeline and tests are available.

In [3]:
import subprocess, sys
from pathlib import Path

# repo_dir is set by the env setup cell above.
# Fallback to CWD so this cell is also safe to run standalone.
try:
    _repo = repo_dir  # type: ignore[name-defined]
except NameError:
    _repo = Path.cwd()

# Install dev extras (pytest, respx, etc.) using the same absolute-path pattern
# as the env setup cell to avoid CWD ambiguity.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", f"{_repo}[dev]", "-q"],
    check=True,
)
print("Dev dependencies installed ✓")

Dev dependencies installed ✓


## Step 2b — Verify the source registry and optionally run the full streaming pipeline

Before launching the pipeline, inspect the checked-in `SOURCES.md` file rather than overwriting it from the notebook.
This keeps the runtime aligned with the repository and avoids accidental drift between notebook content and the canonical source registry.


In [ ]:
from pathlib import Path

sources_path = Path("SOURCES.md")
if not sources_path.exists():
    print("SOURCES.md not found in the project root")
else:
    text = sources_path.read_text(encoding="utf-8")
    lines = text.splitlines()
    print(f"Using checked-in source registry: {sources_path} ({len(lines)} lines)")
    print()
    print("\n".join(lines[:80]))
    if len(lines) > 80:
        print("\n... (truncated preview) ...")


In [6]:
# Full streaming pipeline — DirectoryAgent + SearchAgent run in parallel and
# feed ValidationAgent via asyncio.Queue so validation overlaps with discovery.
# Equivalent to: wcd-pipeline --tier 1
!python scripts/run_pipeline.py --tier 1

2026-03-19 17:51:06.916 | INFO     | webcam_discovery.pipeline:run_pipeline:128 - Pipeline starting — tier=1, mode=streaming-parallel
2026-03-19 17:51:07.853 | INFO     | webcam_discovery.pipeline:run_pipeline:139 - Step 1+2 — Discovery (parallel) overlapping with Validation (batch_size=100, queue_maxsize=500)
2026-03-19 17:51:07.865 | INFO     | webcam_discovery.agents.directory_crawler:_parse:185 - SourcesRegistry: loaded tiers {1: 2, 2: 16, 3: 24, 4: 7, 5: 2} (51 total sources), 13 blocked domains from SOURCES.md
2026-03-19 17:51:07.865 | INFO     | webcam_discovery.agents.directory_crawler:stream:421 - DirectoryAgent.stream: tier=1 → 2 sources
2026-03-19 17:51:07.867 | INFO     | webcam_discovery.agents.directory_crawler:_parse:185 - SourcesRegistry: loaded tiers {1: 2, 2: 16, 3: 24, 4: 7, 5: 2} (51 total sources), 13 blocked domains from SOURCES.md
2026-03-19 17:51:08.401 | INFO     | webcam_discovery.agents.directory_crawler:stream:454 - DirectoryAgent.stream: crawling skylineweb

## Step 3a — Run DirectoryAgent (Tier 1)

Crawls all Tier 1 sources from `SOURCES.md` and records **all discovered camera candidate links**.
Direct `.m3u8` HLS stream URLs are preserved for downstream validation, while other link formats
are retained in the discovery output for inspection and logging.

> **In the full streaming pipeline** (`run_pipeline.py`), `DirectoryAgent` runs
> concurrently with `SearchAgent` using `asyncio.gather`. Both emit candidates
> via their `.stream()` async generator into a shared `asyncio.Queue`, but only
> `.m3u8` URLs are forwarded to `ValidationAgent`; non-HLS candidates are logged
> to `candidates/discovery_non_hls.jsonl`.
> Running this cell on its own executes `DirectoryAgent` in isolation.

### Crawler behaviour

- **`DirectoryTraversalSkill`** — follows sub-category links up to `max_depth=5` to reach
  individual camera pages (e.g. `/en/usa/new-york/niagara.html` is 4 segments deep)
- **`FeedExtractionSkill`** — scans static HTML for direct stream links and other embedded camera references
- **No discovery prefiltering by link type** — discovery keeps `.m3u8`, MJPEG, embed pages, and other camera URLs so operators can inspect all discovered formats
- **`DirectoryAgent.stream()`** — yields candidates batch-by-batch (`BATCH_SIZE=5` sources
  at a time) so the pipeline can validate early results while later sources are still being crawled
- **robots.txt checks run concurrently** — all source domains are checked in parallel
  before crawling begins, so blocked domains add no serial latency
- `MAX_PAGES_PER_SOURCE = 100` guard prevents runaway crawls


In [ ]:
!python scripts/run_discovery.py --tier 1 --output candidates/candidates.jsonl


## Step 3b — Run SearchAgent (DuckDuckGo discovery)

`SearchAgent` complements directory crawling with structured multi-language queries
against DuckDuckGo (no API key required). It generates queries for each Tier-N city
and keeps all public camera-result URLs that are not on the blocked-domain list.

Results are written to `candidates/search_candidates.jsonl`, then merged (deduplicated
by URL) with the directory-crawl results in Step 3c below.

> **In the full streaming pipeline** (`run_pipeline.py`), `SearchAgent` runs
> concurrently with `DirectoryAgent` via `asyncio.gather`. Its `.stream()` async
> generator emits candidates city-by-city directly into the shared `asyncio.Queue`,
> but only `.m3u8` URLs are forwarded to validation; the rest stay in discovery logs.
> Running this cell on its own executes `SearchAgent` in isolation.

### Search behaviour

- **`QueryGenerationSkill`** — generates query templates in English, Japanese, German,
  French, Spanish, Korean, Swedish, and Norwegian for each city
- **`SearchAgent.stream()`** — yields candidates city-by-city so the pipeline receives
  results incrementally rather than waiting for all cities to be searched
- **DuckDuckGo HTML endpoint** — no API key, polite 0.5 s delay between requests,
  concurrency capped at 5 simultaneous searches
- **`MAX_QUERIES_PER_CITY = 3`** guard avoids rate-limit triggers
- Only blocked domains (e.g. `insecam.org`, `shodan.io`) are filtered out before results are written


In [ ]:
!python scripts/run_search.py --tier 1 --output candidates/search_candidates.jsonl

## Step 3c — Merge discovery outputs and prepare validation input

Combines `candidates/candidates.jsonl` (DirectoryAgent) and `candidates/search_candidates.jsonl`
(SearchAgent) into a single deduplicated discovery file, then splits that merged set into:

- `candidates/candidates.jsonl` — **all** discovered candidate links for operator review
- `candidates/validation_candidates.jsonl` — only direct `.m3u8` links that should proceed to `ValidationAgent`
- `candidates/non_hls_candidates.jsonl` — non-`.m3u8` candidates logged at the discovery boundary

> **In the full streaming pipeline** (`run_pipeline.py`), cross-agent URL deduplication is handled inside
> `ValidationAgent.run_from_queue()`, and non-HLS candidates are logged to `candidates/discovery_non_hls.jsonl`
> instead of being forwarded into validation.


In [ ]:
import json, re
from pathlib import Path

_HLS_RE = re.compile(r"\.m3u8(\?|$)", re.IGNORECASE)

dir_path        = Path("candidates/candidates.jsonl")
search_path     = Path("candidates/search_candidates.jsonl")
merged_path     = Path("candidates/candidates.jsonl")
validation_path = Path("candidates/validation_candidates.jsonl")
non_hls_path    = Path("candidates/non_hls_candidates.jsonl")

dir_lines    = dir_path.read_text().splitlines()    if dir_path.exists()    else []
search_lines = search_path.read_text().splitlines() if search_path.exists() else []

seen_urls: set[str] = set()
merged_lines: list[str] = []
validation_lines: list[str] = []
non_hls_lines: list[str] = []

for line in dir_lines + search_lines:
    if not line.strip():
        continue
    try:
        payload = json.loads(line)
        url = payload.get("url", "")
        if not url or url in seen_urls:
            continue
        seen_urls.add(url)
        merged_lines.append(json.dumps(payload, ensure_ascii=False))
        if url.lower().startswith(("http://", "https://")) and _HLS_RE.search(url):
            validation_lines.append(json.dumps(payload, ensure_ascii=False))
        else:
            non_hls_lines.append(json.dumps(payload, ensure_ascii=False))
    except json.JSONDecodeError:
        continue

merged_path.write_text("\n".join(merged_lines), encoding="utf-8")
validation_path.write_text("\n".join(validation_lines), encoding="utf-8")
non_hls_path.write_text("\n".join(non_hls_lines), encoding="utf-8")

print(f"Directory crawl          : {len(dir_lines)} candidates")
print(f"Search results           : {len(search_lines)} candidates")
print(f"Merged discovery total   : {len(merged_lines)} candidates → {merged_path}")
print(f"Validation (.m3u8 only)  : {len(validation_lines)} candidates → {validation_path}")
print(f"Logged non-.m3u8 links   : {len(non_hls_lines)} candidates → {non_hls_path}")


## Step 4 — Inspect discovery outputs

Preview candidate counts broken down by URL type. Discovery intentionally keeps **all** camera-link formats.
Only direct `.m3u8` HLS URLs are written to `validation_candidates.jsonl` and allowed to proceed into Step 5.

The standard HLS playlist extension is `.m3u8` — if you see `.h3u8`, treat that as a typo or malformed candidate rather than a valid stream type.

| Type | Meaning | Handling in Step 5 |
|------|---------|---------------------|
| `hls-stream`    | `.m3u8` HLS playlist URL             | ✅ Forwarded to validation |
| `html-page`     | Camera page URL (no `.m3u8` in URL)  | 📝 Logged at discovery only |
| `embed-page`    | Third-party player iframe URL        | 📝 Logged at discovery only |
| `youtube-embed` | YouTube nocookie embed URL           | 📝 Logged at discovery only |
| `mp4-file`      | Static MP4 video recording           | 📝 Logged at discovery only |
| `invalid-protocol` | Broken or protocol-relative URL    | 📝 Logged at discovery only unless normalization fixes it |


In [ ]:
import json, re
from pathlib import Path
from collections import Counter

all_path        = Path("candidates/candidates.jsonl")
validation_path = Path("candidates/validation_candidates.jsonl")
non_hls_path    = Path("candidates/non_hls_candidates.jsonl")

if not all_path.exists():
    print("candidates.jsonl not found - did the crawler run successfully?")
else:
    lines = all_path.read_text().splitlines()
    candidates = [json.loads(l) for l in lines if l.strip()]

    _hls_re     = re.compile(r"\.m3u8(\?|$)", re.IGNORECASE)
    _mp4_re     = re.compile(r"\.(mp4|webm|ogv)(\?|$)", re.IGNORECASE)
    _youtube_re = re.compile(r"youtube(?:-nocookie)?\.com/embed/", re.IGNORECASE)
    _embed_re   = re.compile(r"/embed[/\?]|embed\.", re.IGNORECASE)

    def url_type(url):
        if _hls_re.search(url):                                 return "hls-stream"
        if _mp4_re.search(url):                                 return "mp4-file"
        if _youtube_re.search(url):                             return "youtube-embed"
        if _embed_re.search(url):                               return "embed-page"
        if not url.lower().startswith(("http://","https://")): return "invalid-protocol"
        return "html-page"

    types   = Counter(url_type(c["url"]) for c in candidates)
    sources = Counter(c.get("source_directory", "unknown") for c in candidates)
    search_count = sum(n for s, n in sources.items() if s.startswith("search:"))
    crawl_count  = len(candidates) - search_count
    validation_count = sum(1 for _ in validation_path.read_text().splitlines() if _.strip()) if validation_path.exists() else 0
    non_hls_count = sum(1 for _ in non_hls_path.read_text().splitlines() if _.strip()) if non_hls_path.exists() else 0

    print(f"Total discovered candidates : {len(candidates)}")
    print(f"  From directory crawl     : {crawl_count}")
    print(f"  From search results      : {search_count}")
    print(f"  Forwarded to validation  : {validation_count}")
    print(f"  Logged at discovery only : {non_hls_count}")
    print()
    print("-- URL type breakdown --")
    print(f"  hls-stream       : {types['hls-stream']:>4}  ✅ forwarded to validation")
    print(f"  html-page        : {types['html-page']:>4}  📝 logged only")
    print(f"  embed-page       : {types['embed-page']:>4}  📝 logged only")
    print(f"  youtube-embed    : {types['youtube-embed']:>4}  📝 logged only")
    print(f"  mp4-file         : {types['mp4-file']:>4}  📝 logged only")
    print(f"  invalid-protocol : {types['invalid-protocol']:>4}  📝 logged unless normalized upstream")
    print()
    print("-- Top source directories --")
    for src, n in sources.most_common(15):
        print(f"  {n:>4}  {src}")
    print()
    print("-- First 10 discovery candidates --")
    for c in candidates[:10]:
        t = url_type(c["url"])
        mark = "✅" if t == "hls-stream" else "📝"
        print(f"  {mark} [{t:<16}]  {c['url']}")
        print(f"               label={c.get('label','-')}  city={c.get('city','-')}, {c.get('country','-')}")
        print()


## Step 5 — Validate `.m3u8` candidates only

`ValidationAgent` deep-probes only the direct `.m3u8` URLs prepared in `validation_candidates.jsonl`
and uses **ffprobe** plus **ffmpeg** frame sampling to confirm the stream is genuinely active
(real video frames present).

### Discovery-to-validation boundary

- Discovery keeps all candidate link formats for operator review
- Only direct `.m3u8` links are forwarded into Step 5
- Non-HLS candidates remain logged in `non_hls_candidates.jsonl` and do not go further for now

| URL type | Outcome |
|----------|---------|
| `.m3u8` with valid `#EXTM3U` | ✅ Probed; status set by ffprobe/ffmpeg |
| `.m3u8` missing protocol (`//…`) | ✅ Normalized during extraction when possible; otherwise ❌ excluded from validation input |
| HTML page / embed page | 📝 Logged at discovery only |
| YouTube / MP4 / MJPEG | 📝 Logged at discovery only |
| `.h3u8` | 📝 Invalid HLS extension; treat as malformed discovery output |

### ffprobe frame-level stream status (`use_ffprobe_validation=True`, default)

After HTTP confirmation that the `.m3u8` playlist is reachable, **ffprobe** inspects the stream and
**ffmpeg** decodes a short sample of frames for image analysis.


In [ ]:
!python scripts/run_validation.py --input candidates/validation_candidates.jsonl --output candidates/validated.jsonl


## Step 6 — Inspect validated cameras

Pipeline funnel, stream-status breakdown, and all validated HLS stream records.
Every retained URL in the output is a directly playable `.m3u8` stream — no user interaction required.

Stream status is set by **ffprobe/ffmpeg** frame analysis (when `WCD_USE_FFPROBE_VALIDATION=true`,
the default). Each record carries one of three statuses:

| `status`  | Meaning |
|-----------|---------|
| `live`    | ffprobe decoded frames with real content and motion detected |
| `unknown` | ffprobe decoded frames but they were blank or frozen, OR HTTP probe was inconclusive |
| `dead`    | ffprobe could not fetch segments, OR the stream failed definitively |

All three statuses are preserved in `validated.jsonl`. Dead streams should stay in the review queue unless they are explicitly re-validated and approved for removal.


In [ ]:
import json, re
from pathlib import Path
from collections import Counter

validated_path  = Path("candidates/validated.jsonl")
candidates_path = Path("candidates/candidates.jsonl")

if not validated_path.exists():
    print("validated.jsonl not found - did validation run successfully?")
else:
    records = [json.loads(l) for l in validated_path.read_text().splitlines() if l.strip()]
    n_cands = sum(1 for l in candidates_path.read_text().splitlines() if l.strip()) \
              if candidates_path.exists() else "?"

    located   = [r for r in records if r.get("latitude") is not None]
    unlocated = [r for r in records if r.get("latitude") is None]

    _VALID_FEED_TYPES = {"HLS_master", "HLS_stream", "MJPEG"}

    _hls_re   = re.compile(r"\.m3u8(\?|$)", re.IGNORECASE)
    _mjpeg_re = re.compile(r"\.(mjpg|mjpeg)(\?|$)", re.IGNORECASE)

    def stream_category(r):
        url = r.get("url", "")
        ft  = r.get("feed_type", "")
        if _hls_re.search(url) or ft in ("HLS_master", "HLS_stream"): return "hls-stream ✅"
        if _mjpeg_re.search(url) or ft == "MJPEG":                    return "mjpeg-stream ✅"
        return f"non-stream ❌ [{ft}]"

    categories = Counter(stream_category(r) for r in records)
    valid_count   = sum(n for k, n in categories.items() if "✅" in k)
    invalid_count = sum(n for k, n in categories.items() if "❌" in k)
    status_counts = Counter(r.get("status", "unknown") for r in records)

    print("-- Pipeline funnel ------------------------------------------")
    print(f"  Candidates (discovery + search) : {n_cands}")
    print(f"  Validated records               : {len(records)}")
    print(f"    with coordinates              : {len(located)}")
    print(f"    without coords (included)     : {len(unlocated)}")
    print(f"    active stream URLs            : {valid_count}  ✅")
    print(f"    non-stream URLs (bug!)        : {invalid_count}  ❌")
    print()
    if invalid_count > 0:
        print("  ⚠️  Non-stream records found — check validation logic.")
        print()

    print("-- Stream status breakdown -------------------------------")
    for status, n in status_counts.most_common():
        print(f"  {status:<12} : {n}")
    print()
    print("-- Stream type breakdown ------------------------------------")
    for cat, n in categories.most_common():
        print(f"  {cat:<30} : {n}")
    print()
    print("-- By feed type ----------------------------------------------")
    for ft, n in Counter(r.get("feed_type", "unknown") for r in records).most_common():
        mark = "✅" if ft in _VALID_FEED_TYPES else "❌"
        print(f"  {mark} {ft:<18} : {n}")
    print()
    print("-- First 20 records -------------------------------------------")
    for r in records[:20]:
        print(f"  [{r.get('status','?'):<7}] [{r.get('feed_type','?'):<10}] {r.get('label','-')}")
        print(f"      {r.get('city','-')}, {r.get('country','-')}")
        print(f"      {r.get('url','')}")
        print()


## Step 7 — Build catalog (CatalogAgent)

`CatalogAgent` takes the validated records and:
1. **Deduplicates** — merges records with identical or near-identical stream URLs
2. **Exports** `camera.geojson` — GeoJSON FeatureCollection (RFC 7946 `[lon, lat]`)
3. **Exports** `cameras.md` — human-readable camera link list
4. **Snapshots** — writes a dated copy to `snapshots/camera_YYYY-MM-DD.geojson`

Both output files are written to the project root so that `map.html` can auto-load `camera.geojson`.

In [ ]:
!python scripts/run_catalog.py --input candidates/validated.jsonl --output .

## Step 8 — Inspect catalog outputs

Summary of `camera.geojson` features, coordinate coverage, and a preview of `cameras.md`.
Serve `map.html` locally to view the interactive map (camera.geojson must be co-located).

In [ ]:
import json
from pathlib import Path
from collections import Counter

geojson_path = Path("camera.geojson")
md_path      = Path("cameras.md")

if not geojson_path.exists():
    print("camera.geojson not found — did the catalog step run successfully?")
else:
    gj = json.loads(geojson_path.read_text())
    features = gj.get("features", [])

    located   = [f for f in features if f["geometry"] is not None]
    unlocated = [f for f in features if f["geometry"] is None]
    countries = Counter(
        f["properties"].get("country", "unknown") for f in features
    )
    feed_types = Counter(
        f["properties"].get("feed_type", "unknown") for f in features
    )
    sources = Counter(
        f["properties"].get("source_directory", "unknown") for f in features
    )

    print("── camera.geojson ───────────────────────────────────────")
    print(f"  Total features          : {len(features)}")
    print(f"  With coordinates        : {len(located)}")
    print(f"  Without coordinates     : {len(unlocated)}")
    print()
    print("── Feed types ───────────────────────────────────────────")
    for ft, n in feed_types.most_common():
        print(f"  {ft:<20} : {n}")
    print()
    print("── Top countries ────────────────────────────────────────")
    for country, n in countries.most_common(15):
        print(f"  {country:<30} : {n}")
    print()
    print("── Top source directories ───────────────────────────────")
    for src, n in sources.most_common(10):
        print(f"  {n:>4}  {src}")
    print()
    print("── First 5 features ─────────────────────────────────────")
    for f in features[:5]:
        p = f["properties"]
        geo = f["geometry"]
        coords = f"{geo['coordinates'][1]:.3f},{geo['coordinates'][0]:.3f}" if geo else "no-coords"
        print(f"  [{p.get('feed_type','?'):<14}] [{coords}]")
        print(f"    {p.get('label','-')} — {p.get('city','-')}, {p.get('country','-')}")
        print(f"    {p.get('url','')}")
        print()

    if md_path.exists():
        lines = md_path.read_text().splitlines()
        print(f"── cameras.md ({len(lines)} lines) — first 20 ──────────────────")
        print("\n".join(lines[:20]))

    print()
    print("── To view the interactive map ──────────────────────────")
    print("  python -m http.server 8000")
    print("  open http://localhost:8000/map.html")

## Step 9 — Export intermediate datasets as GeoJSON

Three supplementary GeoJSON exports for debugging and cross-pipeline comparison.
All three reuse `DeduplicationSkill` and `GeoJSONExportSkill` from the catalog layer.

| Output file | Source | Filter | Notes |
|-------------|--------|--------|-------|
| `validated.geojson` | `candidates/validated.jsonl` | all records | Deduped CameraRecord objects with full geo-enrichment |
| `candidates.geojson` | `candidates/candidates.jsonl` | HLS `.m3u8` only | Raw discovery candidates geo-enriched on the fly |
| `search_candidates.geojson` | `candidates/search_candidates.jsonl` | HLS `.m3u8` only | Search candidates geo-enriched on the fly |

**Why filter to HLS-only?** Raw candidates include HTML embed pages and other
non-stream URLs that only become mappable after the validator probes them.
Filtering to `.m3u8` URLs ensures every feature in these GeoJSONs has a confirmed
direct stream URL that plays automatically with no user interaction.

> **Cells 9b and 9c use `await`** — this requires IPython ≥ 7 (Jupyter Lab, Jupyter Notebook 6+,
> Google Colab) which enables top-level `await` inside notebook cells.


In [ ]:
"""
Step 9a — Export validated.geojson.

Loads candidates/validated.jsonl (full CameraRecord objects produced by
ValidationAgent), deduplicates using the same DeduplicationSkill that
CatalogAgent uses, then exports to validated.geojson via GeoJSONExportSkill.

This gives a GeoJSON snapshot of every record that passed validation before
the final catalog deduplication step.
"""
import json
from pathlib import Path
from webcam_discovery.schemas import CameraRecord
from webcam_discovery.skills.catalog import (
    DeduplicationSkill, DeduplicationInput,
    GeoJSONExportSkill, GeoJSONExportInput,
)

input_path  = Path("candidates/validated.jsonl")
output_path = Path("validated.geojson")

if not input_path.exists():
    print(f"{input_path} not found — run Step 5 first")
else:
    records = [
        CameraRecord(**json.loads(line))
        for line in input_path.read_text().splitlines()
        if line.strip()
    ]
    print(f"Loaded {len(records)} validated records")

    # Deduplicate using the same skill CatalogAgent uses
    dedup_skill = DeduplicationSkill()
    catalog: list[CameraRecord] = []
    for record in records:
        result = dedup_skill.run(DeduplicationInput(
            candidate_record=record,
            existing_catalog=catalog,
        ))
        if result.is_duplicate:
            if result.merged_record and result.canonical_record:
                idx = next(
                    (i for i, r in enumerate(catalog) if r.id == result.canonical_record.id),
                    None,
                )
                if idx is not None:
                    catalog[idx] = result.merged_record
        else:
            catalog.append(record)

    print(f"After dedup: {len(catalog)} records ({len(records) - len(catalog)} duplicates removed)")

    result = GeoJSONExportSkill().run(GeoJSONExportInput(cameras=catalog, output_path=output_path))
    print(f"Exported : {result.exported} features → {output_path}")
    print(f"Skipped  : {result.skipped} records (no coordinates)")

In [ ]:
"""
Step 9b — Export candidates.geojson (HLS .m3u8 only).

Filters candidates.jsonl to direct HLS (.m3u8) stream URLs — those already
resolved to a live stream URL by FeedExtractionSkill during directory crawling.
Non-HLS URLs (HTML pages, embed pages, MJPEG, MP4) are excluded here; they
would be dropped anyway by the hls_only filter in Step 5.

Geocoding uses GeoEnrichmentSkill, which calls the Ollama LLM by default
(USE_LLM_GEODECODE=True).  Each camera's location string (label, city, country)
is sent to the configured Ollama model and returns location, latitude, longitude,
country, region, and continent.  Results are cached process-wide.
"""
import asyncio, json, re
from pathlib import Path
from slugify import slugify
from webcam_discovery.schemas import CameraCandidate, CameraRecord
from webcam_discovery.skills.catalog import (
    GeoEnrichmentSkill, GeoEnrichmentInput,
    GeoJSONExportSkill, GeoJSONExportInput,
)

_HLS_RE = re.compile(r'\.m3u8(\?|$)', re.IGNORECASE)

input_path  = Path("candidates/candidates.jsonl")
output_path = Path("candidates.geojson")

if not input_path.exists():
    print(f"{input_path} not found — run Step 3a first")
else:
    all_cands = [
        CameraCandidate(**json.loads(line))
        for line in input_path.read_text().splitlines()
        if line.strip()
    ]

    hls_cands = [
        c for c in all_cands
        if _HLS_RE.search(c.url) and c.url.lower().startswith(("http://", "https://"))
    ]
    print(f"candidates.jsonl : {len(all_cands)} total → {len(hls_cands)} HLS (.m3u8) streams")

    geo_skill = GeoEnrichmentSkill()

    async def _enrich_all(cands):
        tasks = [
            geo_skill.run(GeoEnrichmentInput(
                city    = c.city    if c.city    and c.city    != "Unknown" else None,
                country = c.country if c.country and c.country != "Unknown" else None,
                label   = c.label   if c.label   and c.label   != c.city    else None,
                url     = c.url,
            ))
            for c in cands
        ]
        return await asyncio.gather(*tasks, return_exceptions=True)

    geo_results = await _enrich_all(hls_cands)

    records: list[CameraRecord] = []
    for candidate, geo in zip(hls_cands, geo_results):
        city    = candidate.city    or "Unknown"
        country = candidate.country or "Unknown"
        label   = candidate.label   or city
        lat = lon = continent = region = None
        if isinstance(geo, Exception):
            pass
        elif geo is not None:
            lat, lon  = geo.latitude, geo.longitude
            continent = geo.continent or "Unknown"
            region    = geo.region
            if geo.country:
                country = geo.country

        records.append(CameraRecord(
            id               = slugify(f"{city} {label}", max_length=80, word_boundary=True, separator="-") or "camera",
            label            = label,
            city             = city,
            country          = country,
            continent        = continent or "Unknown",
            region           = region,
            latitude         = lat,
            longitude        = lon,
            url              = candidate.url,
            feed_type        = "HLS_stream",
            source_directory = candidate.source_directory,
            source_refs      = candidate.source_refs or [candidate.url],
            legitimacy_score = "low",
            status           = "unknown",
            notes            = candidate.notes,
        ))

    located = sum(1 for r in records if r.latitude is not None)
    print(f"Geo-enriched: {located}/{len(records)} records have coordinates")

    result = GeoJSONExportSkill().run(GeoJSONExportInput(cameras=records, output_path=output_path))
    print(f"Exported : {result.exported} features → {output_path}")
    print(f"Skipped  : {result.skipped} records (no coordinates)")


In [ ]:
"""
Step 9c — Export search_candidates.geojson (HLS .m3u8 only).

Filters search_candidates.jsonl to direct HLS (.m3u8) stream URLs,
geo-enriches each record using GeoEnrichmentSkill, then exports via
GeoJSONExportSkill.

Geocoding uses the Ollama LLM by default (USE_LLM_GEODECODE=True): each
camera's location string (label, city, country) is sent to the configured
Ollama model and returns location, latitude, longitude, country, region,
and continent.  Results are cached process-wide.

Note: SearchAgent returns DuckDuckGo result pages, so most URLs are HTML
pages rather than direct stream links.  HLS hits here are rare but valuable.
"""
import asyncio, json, re
from pathlib import Path
from slugify import slugify
from webcam_discovery.schemas import CameraCandidate, CameraRecord
from webcam_discovery.skills.catalog import (
    GeoEnrichmentSkill, GeoEnrichmentInput,
    GeoJSONExportSkill, GeoJSONExportInput,
)

_HLS_RE = re.compile(r'\.m3u8(\?|$)', re.IGNORECASE)

input_path  = Path("candidates/search_candidates.jsonl")
output_path = Path("search_candidates.geojson")

if not input_path.exists():
    print(f"{input_path} not found — run Step 3b first")
else:
    all_cands = [
        CameraCandidate(**json.loads(line))
        for line in input_path.read_text().splitlines()
        if line.strip()
    ]

    hls_cands = [
        c for c in all_cands
        if _HLS_RE.search(c.url) and c.url.lower().startswith(("http://", "https://"))
    ]
    print(f"search_candidates.jsonl : {len(all_cands)} total → {len(hls_cands)} HLS (.m3u8) streams")

    geo_skill = GeoEnrichmentSkill()

    async def _enrich_all(cands):
        tasks = [
            geo_skill.run(GeoEnrichmentInput(
                city    = c.city    if c.city    and c.city    != "Unknown" else None,
                country = c.country if c.country and c.country != "Unknown" else None,
                label   = c.label   if c.label   and c.label   != c.city    else None,
                url     = c.url,
            ))
            for c in cands
        ]
        return await asyncio.gather(*tasks, return_exceptions=True)

    geo_results = await _enrich_all(hls_cands)

    records: list[CameraRecord] = []
    for candidate, geo in zip(hls_cands, geo_results):
        city    = candidate.city    or "Unknown"
        country = candidate.country or "Unknown"
        label   = candidate.label   or city
        lat = lon = continent = region = None
        if isinstance(geo, Exception):
            pass
        elif geo is not None:
            lat, lon  = geo.latitude, geo.longitude
            continent = geo.continent or "Unknown"
            region    = geo.region
            if geo.country:
                country = geo.country

        records.append(CameraRecord(
            id               = slugify(f"{city} {label}", max_length=80, word_boundary=True, separator="-") or "camera",
            label            = label,
            city             = city,
            country          = country,
            continent        = continent or "Unknown",
            region           = region,
            latitude         = lat,
            longitude        = lon,
            url              = candidate.url,
            feed_type        = "HLS_stream",
            source_directory = candidate.source_directory,
            source_refs      = candidate.source_refs or [candidate.url],
            legitimacy_score = "low",
            status           = "unknown",
            notes            = candidate.notes,
        ))

    located = sum(1 for r in records if r.latitude is not None)
    print(f"Geo-enriched: {located}/{len(records)} records have coordinates")

    result = GeoJSONExportSkill().run(GeoJSONExportInput(cameras=records, output_path=output_path))
    print(f"Exported : {result.exported} features → {output_path}")
    print(f"Skipped  : {result.skipped} records (no coordinates)")


# Batch Camera Scanner README

## Overview

This project provides a Python-based batch scanner for public live
camera URLs. It is designed to evaluate whether a camera or stream is:

-   active_streaming
-   active_blank
-   disabled
-   does_not_exist

The scanner supports both:

-   direct media URLs such as .m3u8, .mp4, .mjpeg
-   page URLs that may contain embedded camera players

It is hardened for use in constrained environments such as Google Colab
and supports optional browser-based discovery with automatic fallback
behavior.

------------------------------------------------------------------------

## Features

-   Batch scanning from .jsonl input
-   Accepts one candidate object per line, or JSON arrays of candidate
    objects
-   Handles direct media URLs without requiring browser automation
-   Optional Playwright-based browser discovery for page URLs
-   Automatic fallback to lightweight HTML parsing when Playwright is
    unavailable
-   Safer defaults for Google Colab
-   Outputs both CSV and JSONL audit records
-   Optional browser discovery disable flag for maximum Colab stability

------------------------------------------------------------------------

## Status Definitions

active_streaming: camera reachable, frames decode, meaningful content\
active_blank: frames decode but blank, black, or frozen\
disabled: resource exists but not usable publicly\
does_not_exist: resource missing or invalid

------------------------------------------------------------------------

## Input Format

.jsonl file where each line is:

{"url":"https://example.com/cam.m3u8","label":"Example Cam"}

or a list:

\[{"url":"https://example.com/cam1"},{"url":"https://example.com/cam2"}\]

------------------------------------------------------------------------

## Output Files

results.csv → flattened results\
results.jsonl → full structured audit

------------------------------------------------------------------------

## Core Detection Logic

1.  HTTP reachability\
2.  Media detection\
3.  ffprobe validation\
4.  Frame sampling via ffmpeg\
5.  Frame analysis (brightness, entropy, motion)

------------------------------------------------------------------------

## Google Colab Setup

!apt-get update -y\
!apt-get install -y ffmpeg\
!pip install -q playwright pandas requests opencv-python numpy\
!playwright install --with-deps chromium

------------------------------------------------------------------------

## Running

python batch_camera_scanner_colab.py --input candidates.jsonl --workers
1

Stable mode:

python batch_camera_scanner_colab.py --input candidates.jsonl --workers
1 --disable-browser-discovery

------------------------------------------------------------------------

## Arguments

--input (required)\
--output-csv\
--output-jsonl\
--workers (default 1)\
--disable-browser-discovery\
--no-playwright-fallback

------------------------------------------------------------------------

## Output Fields

url, status, detail, media_url, http_status_code, frames_decoded,
blank_like, frozen_like

------------------------------------------------------------------------

## Limitations

-   Cannot detect backend camera state\
-   Blank vs disabled may overlap\
-   Some streams require auth/session\
-   Night scenes can appear blank

------------------------------------------------------------------------

## Future Enhancements

-   OCR on frames\
-   LLM-assisted classification\
-   Screenshot capture\
-   Confidence scoring

------------------------------------------------------------------------

## Summary

This scanner provides a robust, Colab-safe way to classify public camera
streams using deterministic analysis with optional browser discovery.


In [ ]:
!apt-get update -y
!apt-get install -y ffmpeg
!pip install -q playwright pandas requests opencv-python numpy
!playwright install --with-deps chromium

In [ ]:
%%writefile batch_camera_scanner_colab.py
"""
batch_camera_scanner_colab.py

Colab-hardened batch scanner for public live camera URLs.

Accepted inputs:
1. JSONL file where each line is either:
   - a candidate dict with at least a "url" field, or
   - a JSON array of candidate dicts

Example line:
{"url":"https://videos-3.earthcam.com/fecnetwork/19704.flv/playlist.m3u8?t=...","label":"Pittsburgh CamPittsburgh, PA","city":"Pittsburgh","country":"Pennsylvania","source_directory":"www.earthcam.com","source_refs":["https://www.earthcam.com/usa/pennsylvania/pittsburgh/","https://www.earthcam.com"],"notes":"embedded_in:https://www.earthcam.com/usa/pennsylvania/pittsburgh/"}

Classifications:
- active_streaming
- active_blank
- disabled
- does_not_exist

Colab hardening:
- safer default workers=1
- optional --disable-browser-discovery
- automatic Playwright fallback:
    * if Playwright is unavailable or browser launch fails, scanner does not crash
    * falls back to non-browser classification path
- direct media URLs skip Playwright entirely
- supports page URLs and media URLs
- writes CSV and JSONL outputs

Recommended Colab setup:
!apt-get update -y
!apt-get install -y ffmpeg
!pip install -q playwright pandas requests opencv-python numpy
!playwright install --with-deps chromium

Usage:
python batch_camera_scanner_colab.py \
  --input candidates.jsonl \
  --output-csv results.csv \
  --output-jsonl results.jsonl \
  --workers 1

Optional:
python batch_camera_scanner_colab.py \
  --input candidates.jsonl \
  --disable-browser-discovery

"""

from __future__ import annotations

import argparse
import concurrent.futures
import json
import math
import re
import subprocess
import tempfile
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import cv2
import numpy as np
import pandas as pd
import requests

try:
    from playwright.sync_api import sync_playwright
    PLAYWRIGHT_IMPORT_OK = True
    PLAYWRIGHT_IMPORT_ERROR = None
except Exception as e:
    sync_playwright = None
    PLAYWRIGHT_IMPORT_OK = False
    PLAYWRIGHT_IMPORT_ERROR = str(e)


MEDIA_HINTS = (
    ".m3u8",
    ".mpd",
    ".mp4",
    ".ts",
    ".mjpeg",
    ".mjpg",
    ".flv",
    ".avi",
    ".mov",
    ".webm",
    ".jpg",
    ".jpeg",
    ".png",
)

DEFAULT_PAGE_TIMEOUT_MS = 30000
DEFAULT_SAMPLE_SECONDS = 8
DEFAULT_SAMPLE_FPS = 1
DEFAULT_HTTP_TIMEOUT = 20
DEFAULT_SCAN_RETRIES = 2
DEFAULT_WORKERS = 1


@dataclass
class ScanConfig:
    page_timeout_ms: int = DEFAULT_PAGE_TIMEOUT_MS
    sample_seconds: int = DEFAULT_SAMPLE_SECONDS
    sample_fps: int = DEFAULT_SAMPLE_FPS
    http_timeout: int = DEFAULT_HTTP_TIMEOUT
    retries: int = DEFAULT_SCAN_RETRIES
    max_media_candidates: int = 10
    disable_browser_discovery: bool = False
    allow_playwright_fallback: bool = True


def looks_like_media_url(url: str) -> bool:
    u = str(url).lower()
    return any(h in u for h in MEDIA_HINTS)


def normalize_url(url: str) -> str:
    url = str(url).strip()
    if not url:
        return url
    if not re.match(r"^https?://", url, flags=re.I):
        url = "https://" + url
    return url


def is_candidate_record(obj: Any) -> bool:
    return isinstance(obj, dict) and "url" in obj


def load_jsonl_candidates(path: str) -> List[Dict[str, Any]]:
    candidates: List[Dict[str, Any]] = []
    with open(path, "r", encoding="utf-8") as f:
        for line_num, line in enumerate(f, start=1):
            raw = line.strip()
            if not raw:
                continue
            try:
                item = json.loads(raw)
            except json.JSONDecodeError as e:
                raise ValueError(f"Invalid JSON on line {line_num}: {e}") from e

            if isinstance(item, list):
                for idx, sub in enumerate(item):
                    if not is_candidate_record(sub):
                        raise ValueError(
                            f"Line {line_num}, element {idx} is not a valid candidate dict with a 'url' field."
                        )
                    candidates.append(dict(sub))
            else:
                if not is_candidate_record(item):
                    raise ValueError(
                        f"Line {line_num} is not a valid candidate dict with a 'url' field."
                    )
                candidates.append(dict(item))
    return candidates


def http_head_or_get(url: str, timeout: int = DEFAULT_HTTP_TIMEOUT) -> Dict[str, Any]:
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/122.0.0.0 Safari/537.36"
        )
    }
    try:
        r = requests.head(url, allow_redirects=True, timeout=timeout, headers=headers)
        if r.status_code >= 400 or not r.headers:
            r = requests.get(url, allow_redirects=True, timeout=timeout, stream=True, headers=headers)
        return {
            "ok": True,
            "status_code": r.status_code,
            "final_url": str(r.url),
            "content_type": r.headers.get("content-type", ""),
        }
    except Exception as e:
        return {"ok": False, "error": str(e)}


def requests_html_media_discovery(page_url: str, timeout: int = DEFAULT_HTTP_TIMEOUT) -> Dict[str, Any]:
    """
    Lightweight fallback when Playwright is unavailable or disabled.
    Attempts to extract media-ish URLs from raw HTML without executing JS.
    """
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/122.0.0.0 Safari/537.36"
        )
    }

    candidates = set()
    page_status = None
    page_title = None
    errors: List[str] = []
    visible_signals: Dict[str, Any] = {
        "video_count": 0,
        "img_count": 0,
        "iframe_count": 0,
        "text_markers": [],
        "discovery_mode": "requests_html_fallback",
    }

    try:
        resp = requests.get(page_url, timeout=timeout, headers=headers)
        page_status = resp.status_code
        html = resp.text or ""

        title_match = re.search(r"<title[^>]*>(.*?)</title>", html, flags=re.I | re.S)
        if title_match:
            page_title = re.sub(r"\s+", " ", title_match.group(1)).strip()

        lower_html = html.lower()
        visible_signals["video_count"] = len(re.findall(r"<video\b", html, flags=re.I))
        visible_signals["img_count"] = len(re.findall(r"<img\b", html, flags=re.I))
        visible_signals["iframe_count"] = len(re.findall(r"<iframe\b", html, flags=re.I))

        markers = []
        for marker in [
            "offline",
            "camera unavailable",
            "stream unavailable",
            "no signal",
            "temporarily unavailable",
            "currently unavailable",
            "not available",
        ]:
            if marker in lower_html:
                markers.append(marker)
        visible_signals["text_markers"] = markers

        for m in re.findall(r'https?://[^"\'\s<>]+', html):
            if looks_like_media_url(m):
                candidates.add(m)

        # src/href/data-* extraction
        for m in re.findall(
            r'''(?:src|href|data-src|data-url|data-stream|poster)\s*=\s*["']([^"']+)["']''',
            html,
            flags=re.I,
        ):
            if looks_like_media_url(m):
                candidates.add(m)
            elif m.startswith("//"):
                absolute = "https:" + m
                if looks_like_media_url(absolute):
                    candidates.add(absolute)

    except Exception as e:
        errors.append(f"requests_html_discovery_error: {e}")

    return {
        "page_status": page_status,
        "page_title": page_title,
        "candidates": sorted(candidates),
        "errors": errors,
        "visible_signals": visible_signals,
    }


def discover_media_candidates_playwright(page_url: str, timeout_ms: int = DEFAULT_PAGE_TIMEOUT_MS) -> Dict[str, Any]:
    candidates = set()
    page_status = None
    page_title = None
    errors: List[str] = []
    visible_signals: Dict[str, Any] = {
        "video_count": 0,
        "img_count": 0,
        "iframe_count": 0,
        "text_markers": [],
        "discovery_mode": "playwright",
    }

    if not PLAYWRIGHT_IMPORT_OK:
        errors.append(f"playwright_import_error: {PLAYWRIGHT_IMPORT_ERROR}")
        return {
            "page_status": page_status,
            "page_title": page_title,
            "candidates": sorted(candidates),
            "errors": errors,
            "visible_signals": visible_signals,
        }

    try:
        with sync_playwright() as p:
            browser = p.chromium.launch(
                headless=True,
                args=["--no-sandbox", "--disable-dev-shm-usage"],
            )
            context = browser.new_context(
                viewport={"width": 1440, "height": 900},
                user_agent=(
                    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                    "AppleWebKit/537.36 (KHTML, like Gecko) "
                    "Chrome/122.0.0.0 Safari/537.36"
                ),
            )
            page = context.new_page()

            def on_response(resp):
                try:
                    u = resp.url
                    ct = (resp.headers or {}).get("content-type", "")
                    if looks_like_media_url(u) or any(
                        x in ct.lower()
                        for x in [
                            "video/",
                            "application/vnd.apple.mpegurl",
                            "application/x-mpegurl",
                            "application/dash+xml",
                            "multipart/x-mixed-replace",
                            "image/",
                        ]
                    ):
                        candidates.add(u)
                except Exception:
                    pass

            page.on("response", on_response)

            try:
                resp = page.goto(page_url, wait_until="networkidle", timeout=timeout_ms)
                if resp:
                    page_status = resp.status
                page_title = page.title()

                try:
                    visible_signals["video_count"] = page.locator("video").count()
                except Exception:
                    pass
                try:
                    visible_signals["img_count"] = page.locator("img").count()
                except Exception:
                    pass
                try:
                    visible_signals["iframe_count"] = page.locator("iframe").count()
                except Exception:
                    pass

                dom_urls = page.eval_on_selector_all(
                    "video, source, img, iframe",
                    """els => els.map(el =>
                        el.currentSrc || el.src || el.getAttribute('src') ||
                        el.getAttribute('data-src') || el.getAttribute('poster')
                    ).filter(Boolean)
                    """,
                )
                for u in dom_urls:
                    if isinstance(u, str) and u.strip():
                        candidates.add(u)

                html = page.content()
                lower_html = html.lower()
                markers = []
                for marker in [
                    "offline",
                    "camera unavailable",
                    "stream unavailable",
                    "no signal",
                    "temporarily unavailable",
                    "currently unavailable",
                    "not available",
                ]:
                    if marker in lower_html:
                        markers.append(marker)
                visible_signals["text_markers"] = markers

                for m in re.findall(r'https?://[^"\'\s<>]+', html):
                    if looks_like_media_url(m):
                        candidates.add(m)

            finally:
                context.close()
                browser.close()

    except Exception as e:
        errors.append(f"playwright_launch_or_runtime_error: {e}")

    return {
        "page_status": page_status,
        "page_title": page_title,
        "candidates": sorted(candidates),
        "errors": errors,
        "visible_signals": visible_signals,
    }


def discover_media_candidates(page_url: str, config: ScanConfig) -> Dict[str, Any]:
    """
    Discovery strategy:
    1. If browser discovery is disabled, use requests HTML fallback only.
    2. Otherwise try Playwright.
    3. If Playwright fails and fallback is allowed, use requests HTML fallback.
    """
    if config.disable_browser_discovery:
        result = requests_html_media_discovery(page_url, timeout=config.http_timeout)
        result["browser_discovery_used"] = False
        result["playwright_available"] = PLAYWRIGHT_IMPORT_OK
        return result

    pw_result = discover_media_candidates_playwright(page_url, timeout_ms=config.page_timeout_ms)
    pw_result["browser_discovery_used"] = True
    pw_result["playwright_available"] = PLAYWRIGHT_IMPORT_OK

    if pw_result.get("candidates"):
        return pw_result

    if pw_result.get("errors") and config.allow_playwright_fallback:
        fallback_result = requests_html_media_discovery(page_url, timeout=config.http_timeout)
        fallback_result["browser_discovery_used"] = False
        fallback_result["playwright_available"] = PLAYWRIGHT_IMPORT_OK
        fallback_result["playwright_errors"] = pw_result.get("errors", [])
        fallback_result["fallback_from"] = "playwright"
        return fallback_result

    return pw_result


def ffprobe_video_stream(url: str, timeout: int = 25) -> Dict[str, Any]:
    cmd = [
        "ffprobe",
        "-v", "error",
        "-print_format", "json",
        "-show_streams",
        "-show_format",
        url,
    ]
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
        if result.returncode != 0:
            return {"ok": False, "error": result.stderr.strip() or "ffprobe failed"}
        data = json.loads(result.stdout)
        video_streams = [s for s in data.get("streams", []) if s.get("codec_type") == "video"]
        return {
            "ok": True,
            "has_video": len(video_streams) > 0,
            "video_streams": video_streams,
            "format": data.get("format", {}),
        }
    except Exception as e:
        return {"ok": False, "error": str(e)}


def sample_frames_ffmpeg(url: str, seconds: int = DEFAULT_SAMPLE_SECONDS, fps: int = DEFAULT_SAMPLE_FPS) -> List[np.ndarray]:
    frames: List[np.ndarray] = []
    with tempfile.TemporaryDirectory() as tmpdir:
        pattern = str(Path(tmpdir) / "frame_%03d.jpg")
        cmd = [
            "ffmpeg",
            "-hide_banner",
            "-loglevel", "error",
            "-y",
            "-i", url,
            "-t", str(seconds),
            "-vf", f"fps={fps}",
            pattern,
        ]
        try:
            result = subprocess.run(cmd, capture_output=True, text=True, timeout=60)
            if result.returncode != 0:
                return frames
            for p in sorted(Path(tmpdir).glob("frame_*.jpg")):
                img = cv2.imread(str(p))
                if img is not None:
                    frames.append(img)
        except Exception:
            return frames
    return frames


def frame_metrics(frame: np.ndarray) -> Dict[str, float]:
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    hist = cv2.calcHist([gray], [0], None, [256], [0, 256]).flatten()
    hist = hist / max(hist.sum(), 1.0)
    entropy = float(-(hist[hist > 0] * np.log2(hist[hist > 0])).sum())
    return {
        "mean": float(gray.mean()),
        "std": float(gray.std()),
        "dark_ratio": float((gray < 12).mean()),
        "entropy": entropy,
        "lap_var": float(cv2.Laplacian(gray, cv2.CV_64F).var()),
    }


def analyze_frames(frames: List[np.ndarray]) -> Dict[str, Any]:
    if not frames:
        return {
            "has_frames": False,
            "blank_like": None,
            "frozen_like": None,
            "stats": [],
            "interframe_diffs": [],
        }

    stats = [frame_metrics(f) for f in frames]

    blank_like_count = 0
    for s in stats:
        if s["dark_ratio"] > 0.90 and s["std"] < 8 and s["entropy"] < 2.0:
            blank_like_count += 1

    diffs: List[float] = []
    if len(frames) >= 2:
        prev = cv2.cvtColor(frames[0], cv2.COLOR_BGR2GRAY)
        for f in frames[1:]:
            cur = cv2.cvtColor(f, cv2.COLOR_BGR2GRAY)
            diffs.append(float(np.mean(cv2.absdiff(prev, cur))))
            prev = cur

    frozen_like = len(diffs) > 0 and max(diffs) < 1.5
    blank_like = blank_like_count >= max(2, math.ceil(len(stats) * 0.6))

    return {
        "has_frames": True,
        "blank_like": blank_like,
        "frozen_like": frozen_like,
        "stats": stats,
        "interframe_diffs": diffs,
    }


def summarize_analysis(analysis: Dict[str, Any]) -> Dict[str, Any]:
    stats = analysis.get("stats") or []
    diffs = analysis.get("interframe_diffs") or []

    if not stats:
        return {
            "frames_decoded": 0,
            "blank_like": analysis.get("blank_like"),
            "frozen_like": analysis.get("frozen_like"),
            "mean_brightness_avg": None,
            "std_avg": None,
            "entropy_avg": None,
            "interframe_diff_max": None,
            "interframe_diff_avg": None,
        }

    return {
        "frames_decoded": len(stats),
        "blank_like": analysis.get("blank_like"),
        "frozen_like": analysis.get("frozen_like"),
        "mean_brightness_avg": float(np.mean([s["mean"] for s in stats])),
        "std_avg": float(np.mean([s["std"] for s in stats])),
        "entropy_avg": float(np.mean([s["entropy"] for s in stats])),
        "interframe_diff_max": float(max(diffs)) if diffs else None,
        "interframe_diff_avg": float(np.mean(diffs)) if diffs else None,
    }


def classify_media_url(target_url: str, config: ScanConfig) -> Dict[str, Any]:
    reach = http_head_or_get(target_url, timeout=config.http_timeout)

    if not reach.get("ok"):
        return {
            "status": "does_not_exist",
            "detail": f"Target not reachable: {reach.get('error')}",
            "page_url": target_url,
            "media_url": target_url,
            "reachability": reach,
        }

    if reach.get("status_code") in (404, 410):
        return {
            "status": "does_not_exist",
            "detail": f"Target returned {reach['status_code']}",
            "page_url": target_url,
            "media_url": target_url,
            "reachability": reach,
        }

    probe = ffprobe_video_stream(target_url)
    if not probe.get("ok") or not probe.get("has_video"):
        return {
            "status": "disabled",
            "detail": "Target exists, but no usable video stream could be decoded.",
            "page_url": target_url,
            "media_url": target_url,
            "reachability": reach,
            "probe": probe,
        }

    frames = sample_frames_ffmpeg(target_url, seconds=config.sample_seconds, fps=config.sample_fps)
    analysis = analyze_frames(frames)
    analysis_summary = summarize_analysis(analysis)

    if not analysis["has_frames"]:
        return {
            "status": "disabled",
            "detail": "Media endpoint exists but no decodable frames were obtained.",
            "page_url": target_url,
            "media_url": target_url,
            "reachability": reach,
            "probe": probe,
            "analysis": analysis,
            "analysis_summary": analysis_summary,
        }

    if analysis["blank_like"] or analysis["frozen_like"]:
        return {
            "status": "active_blank",
            "detail": "Camera is reachable and producing frames, but they appear blank or frozen.",
            "page_url": target_url,
            "media_url": target_url,
            "reachability": reach,
            "probe": probe,
            "analysis": analysis,
            "analysis_summary": analysis_summary,
        }

    return {
        "status": "active_streaming",
        "detail": "Camera is reachable and producing changing visual frames.",
        "page_url": target_url,
        "media_url": target_url,
        "reachability": reach,
        "probe": probe,
        "analysis": analysis,
        "analysis_summary": analysis_summary,
    }


def classify_page_url(page_url: str, config: ScanConfig) -> Dict[str, Any]:
    reach = http_head_or_get(page_url, timeout=config.http_timeout)

    if not reach.get("ok"):
        return {
            "status": "does_not_exist",
            "detail": f"Page not reachable: {reach.get('error')}",
            "page_url": page_url,
            "reachability": reach,
        }

    if reach.get("status_code") in (404, 410):
        return {
            "status": "does_not_exist",
            "detail": f"Page returned {reach['status_code']}",
            "page_url": page_url,
            "reachability": reach,
        }

    discovered = discover_media_candidates(page_url, config=config)
    visible_signals = discovered.get("visible_signals", {})
    markers = visible_signals.get("text_markers", [])

    if not discovered.get("candidates"):
        if markers:
            return {
                "status": "disabled",
                "detail": f"Page exists and contains unavailable/offline markers: {', '.join(markers)}",
                "page_url": page_url,
                "reachability": reach,
                "discovery": discovered,
            }

        if discovered.get("errors"):
            return {
                "status": "disabled",
                "detail": "Page exists, but no playable camera resource was discovered in this environment.",
                "page_url": page_url,
                "reachability": reach,
                "discovery": discovered,
            }

        if discovered.get("page_status") and 200 <= discovered["page_status"] < 400:
            return {
                "status": "disabled",
                "detail": "Page exists but no playable camera resource was discovered.",
                "page_url": page_url,
                "reachability": reach,
                "discovery": discovered,
            }

        return {
            "status": "does_not_exist",
            "detail": "No page or camera resource could be confirmed.",
            "page_url": page_url,
            "reachability": reach,
            "discovery": discovered,
        }

    candidates = discovered["candidates"][: config.max_media_candidates]
    last_probe_error: Optional[str] = None

    for candidate in candidates:
        probe = ffprobe_video_stream(candidate)
        if not probe.get("ok"):
            last_probe_error = probe.get("error")
            continue
        if not probe.get("has_video"):
            continue

        frames = sample_frames_ffmpeg(candidate, seconds=config.sample_seconds, fps=config.sample_fps)
        analysis = analyze_frames(frames)
        analysis_summary = summarize_analysis(analysis)

        if not analysis["has_frames"]:
            return {
                "status": "disabled",
                "detail": "Media endpoint exists but no decodable frames were obtained.",
                "page_url": page_url,
                "media_url": candidate,
                "reachability": reach,
                "discovery": discovered,
                "probe": probe,
                "analysis": analysis,
                "analysis_summary": analysis_summary,
            }

        if analysis["blank_like"] or analysis["frozen_like"]:
            return {
                "status": "active_blank",
                "detail": "Camera is reachable and producing frames, but they appear blank or frozen.",
                "page_url": page_url,
                "media_url": candidate,
                "reachability": reach,
                "discovery": discovered,
                "probe": probe,
                "analysis": analysis,
                "analysis_summary": analysis_summary,
            }

        return {
            "status": "active_streaming",
            "detail": "Camera is reachable and producing changing visual frames.",
            "page_url": page_url,
            "media_url": candidate,
            "reachability": reach,
            "discovery": discovered,
            "probe": probe,
            "analysis": analysis,
            "analysis_summary": analysis_summary,
        }

    return {
        "status": "disabled",
        "detail": "Page exists, but no usable public video stream could be decoded."
                  + (f" Last probe error: {last_probe_error}" if last_probe_error else ""),
        "page_url": page_url,
        "reachability": reach,
        "discovery": discovered,
    }


def classify_once(target_url: str, config: ScanConfig) -> Dict[str, Any]:
    if looks_like_media_url(target_url):
        return classify_media_url(target_url, config)
    return classify_page_url(target_url, config)


def classify_with_retries(target_url: str, config: ScanConfig) -> Dict[str, Any]:
    attempts = []
    final_result = None

    for attempt in range(1, config.retries + 2):
        result = classify_once(target_url, config)
        result["attempt"] = attempt
        attempts.append(result)

        if result["status"] == "active_streaming":
            final_result = result
            break

        final_result = result
        time.sleep(1)

    if final_result is None:
        final_result = {
            "status": "does_not_exist",
            "detail": "No result produced.",
            "page_url": target_url,
        }

    final_result["attempts"] = attempts
    return final_result


def flatten_result(result: Dict[str, Any], original_record: Dict[str, Any]) -> Dict[str, Any]:
    discovery = result.get("discovery") or {}
    visible = discovery.get("visible_signals") or {}
    summary = result.get("analysis_summary") or {}
    reach = result.get("reachability") or {}

    flat = dict(original_record)
    flat.update({
        "url": result.get("page_url") or original_record.get("url"),
        "status": result.get("status"),
        "detail": result.get("detail"),
        "media_url": result.get("media_url"),
        "http_ok": reach.get("ok"),
        "http_status_code": reach.get("status_code"),
        "final_url": reach.get("final_url"),
        "content_type": reach.get("content_type"),
        "page_status": discovery.get("page_status"),
        "page_title": discovery.get("page_title"),
        "media_candidates_found": len(discovery.get("candidates") or []),
        "video_count": visible.get("video_count"),
        "img_count": visible.get("img_count"),
        "iframe_count": visible.get("iframe_count"),
        "text_markers": "; ".join(visible.get("text_markers") or []),
        "discovery_mode": visible.get("discovery_mode"),
        "browser_discovery_used": discovery.get("browser_discovery_used"),
        "playwright_available": discovery.get("playwright_available"),
        "playwright_errors": "; ".join(discovery.get("playwright_errors") or discovery.get("errors") or []),
        "frames_decoded": summary.get("frames_decoded"),
        "blank_like": summary.get("blank_like"),
        "frozen_like": summary.get("frozen_like"),
        "mean_brightness_avg": summary.get("mean_brightness_avg"),
        "std_avg": summary.get("std_avg"),
        "entropy_avg": summary.get("entropy_avg"),
        "interframe_diff_max": summary.get("interframe_diff_max"),
        "interframe_diff_avg": summary.get("interframe_diff_avg"),
        "attempt_count": len(result.get("attempts") or []),
    })
    return flat


def process_record(record: Dict[str, Any], config: ScanConfig) -> Tuple[Dict[str, Any], Dict[str, Any]]:
    original = dict(record)
    target_url = normalize_url(original.get("url", ""))

    if not target_url:
        result = {
            "status": "does_not_exist",
            "detail": "Missing URL.",
            "page_url": "",
            "attempts": [],
        }
        return flatten_result(result, original), {"input": original, "result": result}

    result = classify_with_retries(target_url, config)
    flat = flatten_result(result, original)
    raw = {"input": original, "result": result}
    return flat, raw


def run_batch_from_candidates(
    candidates: Sequence[Dict[str, Any]],
    output_csv: str,
    output_jsonl: str,
    workers: int,
    config: ScanConfig,
) -> None:
    candidate_list = [dict(x) for x in candidates]
    if not candidate_list:
        raise ValueError("No candidate records were provided.")
    for idx, item in enumerate(candidate_list):
        if not is_candidate_record(item):
            raise ValueError(f"Candidate at index {idx} is not a dict with a 'url' field.")

    flat_results: List[Dict[str, Any]] = []
    raw_results: List[Dict[str, Any]] = []

    with concurrent.futures.ThreadPoolExecutor(max_workers=workers) as executor:
        futures = [executor.submit(process_record, record, config) for record in candidate_list]

        for i, future in enumerate(concurrent.futures.as_completed(futures), start=1):
            try:
                flat, raw = future.result()
                flat_results.append(flat)
                raw_results.append(raw)
                print(f"[{i}/{len(candidate_list)}] {flat.get('url')} -> {flat.get('status')}")
            except Exception as e:
                err = {
                    "url": None,
                    "status": "error",
                    "detail": str(e),
                }
                flat_results.append(err)
                raw_results.append({"input": None, "result": err})
                print(f"[{i}/{len(candidate_list)}] ERROR -> {e}")

    # Keep deterministic-ish output ordering by original input URL/label when possible
    results_df = pd.DataFrame(flat_results)
    results_df.to_csv(output_csv, index=False)

    with open(output_jsonl, "w", encoding="utf-8") as f:
        for item in raw_results:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

    print(f"\nWrote CSV:   {output_csv}")
    print(f"Wrote JSONL: {output_jsonl}")


def run_batch_from_jsonl(
    input_jsonl: str,
    output_csv: str,
    output_jsonl: str,
    workers: int,
    config: ScanConfig,
) -> None:
    records = load_jsonl_candidates(input_jsonl)
    run_batch_from_candidates(
        candidates=records,
        output_csv=output_csv,
        output_jsonl=output_jsonl,
        workers=workers,
        config=config,
    )


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Colab-hardened batch scanner for public live camera page URLs and direct media URLs."
    )
    parser.add_argument("--input", required=True, help="Input .jsonl path.")
    parser.add_argument("--output-csv", default="results.csv", help="Output CSV path.")
    parser.add_argument("--output-jsonl", default="results.jsonl", help="Output JSONL path.")
    parser.add_argument("--workers", type=int, default=DEFAULT_WORKERS, help="Concurrent workers. Default is 1 for Colab stability.")
    parser.add_argument("--page-timeout-ms", type=int, default=DEFAULT_PAGE_TIMEOUT_MS)
    parser.add_argument("--sample-seconds", type=int, default=DEFAULT_SAMPLE_SECONDS)
    parser.add_argument("--sample-fps", type=int, default=DEFAULT_SAMPLE_FPS)
    parser.add_argument("--http-timeout", type=int, default=DEFAULT_HTTP_TIMEOUT)
    parser.add_argument("--retries", type=int, default=DEFAULT_SCAN_RETRIES)
    parser.add_argument("--max-media-candidates", type=int, default=10)
    parser.add_argument(
        "--disable-browser-discovery",
        action="store_true",
        help="Disable Playwright/browser discovery and use lightweight HTML fallback only for page URLs.",
    )
    parser.add_argument(
        "--no-playwright-fallback",
        action="store_true",
        help="Do not fall back to requests-based HTML discovery when Playwright fails.",
    )
    return parser.parse_args()


def main() -> None:
    args = parse_args()
    config = ScanConfig(
        page_timeout_ms=args.page_timeout_ms,
        sample_seconds=args.sample_seconds,
        sample_fps=args.sample_fps,
        http_timeout=args.http_timeout,
        retries=args.retries,
        max_media_candidates=args.max_media_candidates,
        disable_browser_discovery=args.disable_browser_discovery,
        allow_playwright_fallback=not args.no_playwright_fallback,
    )
    records = load_jsonl_candidates(args.input)
    run_batch_from_candidates(
        candidates=records,
        output_csv=args.output_csv,
        output_jsonl=args.output_jsonl,
        workers=args.workers,
        config=config,
    )


if __name__ == "__main__":
    main()

In [ ]:
!python batch_camera_scanner_colab.py \
  --input candidates/candidates.jsonl \
  --output-csv results.csv \
  --output-jsonl results.jsonl \
  --workers 1 \
  --disable-browser-discovery

In [ ]:
!python batch_camera_scanner_colab.py \
  --input candidates/candidates.jsonl \
  --output-csv results.csv \
  --output-jsonl results.jsonl \
  --workers 1